In [2]:
import os
import os.path as pa
import json
from astropy.table import Table
import numpy as np

In [3]:
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroplan import Observer, FixedTarget
from astroplan import AltitudeConstraint, AtNightConstraint, is_observable
from astroplan.constraints import AirmassConstraint
from astroplan.utils import time_grid_from_range

In [4]:
def peak_half_ctio(ra_deg, dec_deg):
    # CTIO site
    obs = Observer.at_site("Cerro Tololo")
    t0 = Time.now()

    # Get night boundaries
    night_start = obs.twilight_evening_astronomical(t0, which="nearest")
    if obs.sun_altaz(night_start).alt > -18*u.deg:
        night_start = obs.twilight_evening_astronomical(t0, which="previous")
    night_end = obs.twilight_morning_astronomical(night_start + 0.5*u.day, which="nearest")
    night_mid = night_start + 0.5*(night_end - night_start)

    # Target and transit
    target = FixedTarget(coord=SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg))
    transit_time = obs.target_meridian_transit_time(t0, target, which="nearest")

    # Classify half
    if transit_time < night_mid:
        return ["z", "i", "r"]
    else:
        return ["r", "i", "z"]

In [ ]:
eventid = "TDE2026nhm"
target_ra, target_dec = 253.058405,  -4.102048
#filter_order = peak_half_ctio(target_ra, target_dec)
filter_order = ["g", "r", "i", "z"]
print(filter_order)

['g', 'r', 'i', 'z']


In [6]:
exposure_template = {
    "count": 1,
    "object": eventid,
    "program": "DESI Transients Survey",
    "expType": "object",
    "note": "None",
    "comment": "DESI Transients Survey {}".format(eventid),
    "wait": "False",
    "proposer": "Palmese",
    "propid": "2025A-729671"
}

In [1]:
filter_exptimes = {
    "g": 50,
    "r": 70,
    "i": 90,
    "z": 100
}

In [8]:
dither_dec = 0.0825 #always +/- dec
dither_ra = 0.312 / np.cos(np.radians(target_dec))

In [9]:
obs_list = []

for i in range(len(filter_order)):
    f = filter_order[i]

    dither_ra_mult = (i % 3) - 1  # alternate 0, 1, 0, 1
    ra_pointing = target_ra + dither_ra * dither_ra_mult
    dither_dec_mult = 1 if (i // 2) % 2 == 0 else -1  # alternate every two exposures
    dec_pointing = target_dec + dither_dec * dither_dec_mult

    exp = exposure_template.copy()
    exp["filter"] = f
    exp["exptime"] = filter_exptimes[f]

    exp["ra"] = round(ra_pointing, 4)
    exp["dec"] = round(dec_pointing, 4)

    obs_list.append(exp)

In [10]:
date = "260601"
outpath = f"{date}/DESIRT_{eventid}.json"
with open(outpath, "w") as f:
    json.dump(obs_list, f, indent=4)

In [82]:
exp

{'count': 1,
 'object': '2025vjw',
 'program': 'DESI Transients Survey',
 'expType': 'object',
 'note': 'None',
 'comment': 'DESI Transients Survey 2025vjw',
 'wait': 'False',
 'proposer': 'Palmese',
 'propid': '2025A-729671',
 'filter': 'Y',
 'exptime': 250,
 'ra': 347.2533,
 'dec': -7.4363}